# Round 2: backbone screen on fold 0 (token head), including the DINO / DINOv2 / DeiT ViT track
Question: does a different encoder beat the EfficientNet-B0 token model (fold 0: macro 0.786, KEY-4 0.768; one seed) by more than noise,
at a runtime you can afford? Arms: DINOv2-S/14 (168 px), DINO ViT-S/16 (160 px), DeiT-tiny/16, ConvNeXt-T, EfficientNet-B2, plus a
B0 seed-7 repeat as the noise reference. Cell 2 TIMES every arm on your GPU first and only runs the ones whose projected epoch fits
`MAX_EPOCH_MIN`, so a slow backbone cannot eat your quota. ~65 min per B0-sized job; ViTs / ConvNeXt are slower (measured, not guessed).
Inputs: competition, `knee-mri-pipeline` (knee_pipeline_v6, re-uploaded: it now has ViT support + bench_backbones.py), caching output,
training output. GPU T4 x2, internet ON (pretrained weights are downloaded), Save & Run All.

In [1]:
import os, sys, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
CODE   = Path(os.environ.get("KNEE_CODE", "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA", "/kaggle/input/competitions/rsna-knee-abnormality-detection"))

# Inputs can sit under any notebook slug: use the usual path if it exists, otherwise look for them by content.
def _scan(maxdepth=5):
    caches, labels, oofs = [], [], []
    q = [(ROOT, 0)]
    while q:
        d, k = q.pop(0)
        try: ents = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError: continue
        for e in ents:
            p = Path(e.path)
            if e.is_dir(follow_symlinks=False):
                if e.name == "cache":                                   # never walk into the 4,407-file cache folder
                    if (p / "cache_index.csv").exists(): caches.append(p)
                    continue
                if e.name not in {"train_series", "test_series", "__pycache__", ".git"} and k < maxdepth: q.append((p, k + 1))
            elif e.name == "labels_soft.csv": labels.append(p)
            elif e.name == "oof_all.csv": oofs.append(p)
    return caches, labels, oofs
def _resolve(var, env, default, found):
    p = Path(os.environ.get(env, default))
    if p.exists(): return p
    uniq = sorted(set(found))
    print(f"{var}: usual path missing ({p})\n   found by content: {uniq}")
    assert len(uniq) == 1, f"{var}: attach the notebook output that holds it (caching output for CACHE/LABELS, training output for BASE)"
    return uniq[0]
_caches, _labels, _oofs = _scan()
CACHE  = _resolve("CACHE",  "KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache", _caches)
LABELS = _resolve("LABELS", "KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv", _labels)
BASE   = _resolve("BASE",   "KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv", _oofs)

for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("bench_backbones.py", "tests/smoke_v6.py", "geom.py"):
    assert (CODE / f).exists(), f"{f} missing: re-upload knee_pipeline_v6.zip to the pipeline dataset"
assert "vit_size" in (CODE / "model.py").read_text(), "model.py has no ViT support: dataset not updated"
print("paths OK")

paths OK


In [2]:
r = subprocess.run([sys.executable, str(CODE / "tests/smoke_v6.py")], capture_output=True, text=True, cwd=str(CODE))
print(r.stdout[-2500:]); assert r.returncode == 0, r.stderr[-2500:]

head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS



In [3]:
# ---- benchmark every arm: real micro-batch, AMP, this GPU. One process per arm. ----
ARMS = {  # key: (timm name, priority)   lower number runs first
    "effb0_s7": ("efficientnet_b0", 0),                              # noise reference: same as v6_tok, seed 7
    "dinov2":   ("vit_small_patch14_dinov2.lvd142m", 1),
    "dino16":   ("vit_small_patch16_224.dino", 2),
    "convnext": ("convnext_tiny.in12k_ft_in1k", 3),
    "effb2":    ("efficientnet_b2", 4),
    "deit":     ("deit_tiny_patch16_224.fb_in1k", 5),
}
MAX_EPOCH_MIN = 14        # skip an arm whose projected epoch is longer (12 epochs x 14 min = 2.8 h)
MAX_EPOCH_MIN_DINO = 22   # DINO / DINOv2 arms are allowed up to 22 min per epoch (4.4 h)
MAX_GPU_HOURS = 8         # per-GPU queue cap
REF_EPOCH_S   = 390       # measured epoch time of the B0 token runs (377-415 s)
B = {}
for key, (name, _) in ARMS.items():
    if key == "effb0_s7": continue
    for bs in (4, 2):
        r = subprocess.run([sys.executable, str(CODE / "bench_backbones.py"), "--arm", name, "--bs", str(bs)],
                           capture_output=True, text=True, cwd=str(CODE))
        line = [l for l in r.stdout.splitlines() if l.startswith("BENCH_JSON ")]
        res = json.loads(line[-1][11:]) if line else dict(arm=name, bs=bs, status="no output: " + r.stderr[-300:])
        if res["status"] != "oom": break
        print(f"{key}: OOM at bs {bs}, retrying smaller")
    B[key] = res
b0 = subprocess.run([sys.executable, str(CODE / "bench_backbones.py"), "--arm", "efficientnet_b0", "--bs", "4"], capture_output=True, text=True, cwd=str(CODE))
B["effb0"] = json.loads([l for l in b0.stdout.splitlines() if l.startswith("BENCH_JSON ")][-1][11:])
assert B["effb0"]["status"] == "ok", B["effb0"]
B["effb0_s7"] = dict(B["effb0"])
rows = []
for key, res in B.items():
    if res["status"] != "ok":
        rows.append(dict(arm=key, status=res["status"])); continue
    ratio = (res["train_s"] * 4 / res["bs"]) / B["effb0"]["train_s"]
    ep = REF_EPOCH_S * max(1.0, ratio)
    rows.append(dict(arm=key, bs=res["bs"], params_M=res["params_m"], peak_GB=res["peak_gb"], train_s=res["train_s"],
                     slower_than_B0=round(ratio, 2), proj_epoch_min=round(ep / 60, 1), proj_job_h=round((ep * 12 + 150) / 3600, 2),
                     infer_min_5models_1300=round(res["infer_s_per_study"] * 1300 * 5 / 60, 1), status="ok"))
T = pd.DataFrame(rows); print(T.to_string(index=False))

     arm  bs  params_M  peak_GB  train_s  slower_than_B0  proj_epoch_min  proj_job_h  infer_min_5models_1300 status
  dinov2   4     22.52     5.82   0.3685            1.48             9.6        1.97                     4.8     ok
  dino16   4     22.57     3.77   0.2340            0.94             6.5        1.34                     3.1     ok
convnext   4     28.86     6.69   0.3956            1.59            10.3        2.11                     6.2     ok
   effb2   4      8.90     6.65   0.3707            1.49             9.7        1.98                     5.6     ok
    deit   4      6.40     1.91   0.1093            0.44             6.5        1.34                     1.4     ok
   effb0   4      5.18     4.59   0.2486            1.00             6.5        1.34                     4.4     ok
effb0_s7   4      5.18     4.59   0.2486            1.00             6.5        1.34                     4.4     ok


In [4]:
# ---- plan: arms that fit, balanced over the two GPUs ----
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--only_folds", "0", "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16"]
plan = []
for _, r in T[T.status == "ok"].iterrows():
    key = r.arm
    if key == "effb0": continue                      # calibration only
    cap = MAX_EPOCH_MIN_DINO if key in ("dinov2", "dino16") else MAX_EPOCH_MIN       # the DINO track gets a looser cap
    if r.proj_epoch_min > cap and key != "effb0_s7":
        print(f"SKIP {key}: projected epoch {r.proj_epoch_min} min > {cap}"); continue
    bs = int(r.bs); extra = ["--head", "tokens", "--backbone", ARMS[key][0], "--bs", str(bs), "--accum", str(8 // bs)]
    if key == "effb0_s7": extra += ["--seed", "7"]
    plan.append((ARMS[key][1], key, extra, float(r.proj_job_h)))
plan.sort(key=lambda t: -t[3])
load = {0: 0.0, 1: 0.0}; QUEUES = {0: [], 1: []}
for pr, key, extra, h in plan:
    g = min(load, key=load.get)
    if load[g] + h > MAX_GPU_HOURS:
        print(f"SKIP {key}: does not fit in {MAX_GPU_HOURS} h per GPU"); continue
    QUEUES[g].append((f"r2_{key}", extra)); load[g] += h
print({g: [n for n, _ in q] for g, q in QUEUES.items()}, "| projected wall h per GPU:", {g: round(v, 1) for g, v in load.items()})

{0: ['r2_convnext', 'r2_dino16', 'r2_deit'], 1: ['r2_effb2', 'r2_dinov2', 'r2_effb0_s7']} | projected wall h per GPU: {0: 4.8, 1: 5.3}


In [5]:
LOGS = {}
def run_queue(gpu, jobs):
    for name, extra in jobs:
        out = W / name
        if (out / "oof.csv").exists():
            print(f"[{name}] skipped (oof.csv exists)"); continue
        cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + extra + ["--out", str(out)]
        LOGS[name] = W / f"{name}.log"
        print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
        with open(LOGS[name], "w") as fh:
            rc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE)).wait()
        print(f"[{name}] exit code {rc}", flush=True)
threads = [threading.Thread(target=run_queue, args=(g, q)) for g, q in QUEUES.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(900)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f.exists() else []
        last[n] = ls[-1] if ls else "(none yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished")

[r2_effb2] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --head tokens --backbone efficientnet_b2 --bs 4 --accum 2 --out /kaggle/working/r2_effb2
[r2_convnext] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --head tokens --backbone convnext_tiny.in12k_ft_in1k --bs 4 --accum 2 --out /kaggle/working/r2_convnext
[0.25 h] {
 "r2_effb2": "[f0] ep 2/12 loss 0.7820 | val macro-AUC 0.7228 | gold 0.6973 | 436s",
 "r2_convnext": "[f0

In [6]:
# ---- analysis: fold 0, confident old labels ----
from sklearn.metrics import roc_auc_score
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
arms = {n: pd.read_csv(W / n / "oof.csv") for q in QUEUES.values() for n, _ in q if (W / n / "oof.csv").exists()}
assert arms, "no arm finished"
first = next(iter(arms.values()))
val = first[[f"p_{l}" for l in LAB]].notna().all(axis=1).values
assert np.allclose(base.loc[val, [f"y_{l}" for l in LAB]].values, first.loc[val, [f"y_{l}" for l in LAB]].values, atol=1e-6)
def auc(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
P = {"baseline(B0 series)": base}; P.update(arms)
R = pd.DataFrame({n: {l: auc(base.loc[val, f"y_{l}"].values, d.loc[val, f"p_{l}"].values) for l in LAB} for n, d in P.items()})
R.loc["MACRO"] = R.loc[LAB].mean(); R.loc["KEY-4"] = R.loc[KEY4].mean()
print(R.round(3).to_string())
print("\nreferences, fold 0: baseline macro 0.779 / KEY-4 0.755 | v6_tok (B0, seed 42) 0.786 / 0.768 | v6_both seeds 0.790 / 0.801")
ref_macro = 0.786
for n in arms:
    d = R.loc["MACRO", n] - ref_macro
    verdict = "PROMISING (> +0.015 over v6_tok)" if d > 0.015 else ("worse (< -0.015)" if d < -0.015 else "tie within config noise (~0.01)")
    print(f"{n:14s} macro {R.loc['MACRO', n]:.3f} ({d:+.3f} vs v6_tok) | KEY-4 {R.loc['KEY-4', n]:.3f} -> {verdict}")
print("\nA promising arm still needs a 5-fold run; check its inference time in the table above before choosing it.")

                  baseline(B0 series)  r2_convnext  r2_deit  r2_effb2  r2_effb0_s7
ACL                             0.833        0.892    0.815     0.874        0.858
MCL                             0.713        0.805    0.720     0.722        0.727
Medial Meniscus                 0.757        0.870    0.761     0.785        0.778
Lateral Meniscus                0.715        0.776    0.715     0.713        0.718
Medial OA                       0.861        0.874    0.858     0.863        0.865
Lateral OA                      0.749        0.767    0.744     0.728        0.755
PF OA                           0.722        0.740    0.737     0.735        0.733
Effusion                        0.801        0.831    0.816     0.809        0.817
Synovitis                       0.799        0.789    0.780     0.783        0.782
Baker's                         0.747        0.865    0.802     0.796        0.801
Contusion                       0.808        0.834    0.803     0.788        0.827
Frac